Next-Day Rain Forecasting Using Weather Parameters: A Data-Driven Model for Indian Climate Conditions

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
import numpy as np
from sklearn.metrics import f1_score
from sklearn.calibration import CalibratedClassifierCV
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, brier_score_loss,f1_score

df = pd.read_csv("C:\\Users\\as191\\Downloads\\Rainfall_prediction\\India_Capitals_Daily_Weather_years.csv", parse_dates=['Date'])



df = df.sort_values(['City','Date']).reset_index(drop=True)

required_cols = ['City','Date','Max Temp','Min Temp','Mean Temp','Humidity',
                 'Pressure','WindSpeed','CloudCover','Rainfall','RainTomorrow']
df = df[required_cols].dropna().copy()

if df['RainTomorrow'].dtype == object:
    df['RainTomorrow'] = df['RainTomorrow'].map({'Yes':1,'No':0})

print("Class distribution:\n", df['RainTomorrow'].value_counts(normalize=True))


Class distribution:
 RainTomorrow
1    0.567243
0    0.432757
Name: proportion, dtype: float64


In [2]:
df_city = df.copy()

df_city = pd.get_dummies(df_city, columns=["City"], drop_first=False)

print(df_city.shape)
# print(df_city.columns.tolist())


(13176, 44)


In [3]:
df_city = df_city.sort_values('Date').reset_index(drop=True)
df_city['Rain_lag1'] = df_city['Rainfall'].shift(1).fillna(0)

df_city['Rain_3sum'] = df_city['Rainfall'].rolling(3, min_periods=1).sum().shift(1).fillna(0)
print(df_city)
df_city['Temp_range'] = df_city['Max Temp'] - df_city['Min Temp']

df_city['Month'] = df_city['Date'].dt.month
df_city['DayOfYear'] = df_city['Date'].dt.dayofyear
df_city['IsMonsoon'] = df_city['Month'].isin([6,7,8,9]).astype(int)

df_city = df_city.dropna().reset_index(drop=True)


            Date  Max Temp  Min Temp  Mean Temp  Humidity  Pressure  \
0     2024-12-04      28.8      16.2       21.6        71    1010.9   
1     2024-12-04      25.5      15.5       19.9        73    1006.9   
2     2024-12-04      23.6      11.8       17.1        75     925.3   
3     2024-12-04      14.7       0.7        6.0        53     788.9   
4     2024-12-04      26.8      17.3       21.8        56     955.3   
...          ...       ...       ...        ...       ...       ...   
13171 2025-12-04      27.0      17.0       21.8        57    1015.2   
13172 2025-12-04      18.6       7.6       12.3        68     857.3   
13173 2025-12-04      28.3      24.7       26.6        83    1009.9   
13174 2025-12-04      22.1       9.4       15.2        55     993.5   
13175 2025-12-04      31.3      24.4       27.1        79    1009.9   

       WindSpeed  CloudCover  Rainfall  RainTomorrow  ...  City_Port Blair  \
0            9.6           3       0.0             0  ...            

In [4]:



city_cols = [col for col in df_city.columns if col.startswith("City_")]

features = [
    'Max Temp','Min Temp','Mean Temp','Humidity','Pressure','WindSpeed',
    'CloudCover','Rainfall','Rain_lag1','Rain_3sum','Temp_range',
    'Month','DayOfYear','IsMonsoon'
] + city_cols

X = df_city[features]
y = df_city['RainTomorrow']

# X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

print("Train/test sizes:", X_train.shape, X_test.shape)

Train/test sizes: (10540, 48) (2636, 48)


In [ ]:


# Logistic regression pipeline
pipe_log = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))
])


pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', KNeighborsClassifier(n_neighbors=5))
])


pipe_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', XGBClassifier( eval_metric='logloss', n_estimators=200, max_depth=5))
])


In [ ]:




tscv = TimeSeriesSplit(n_splits=4)


log_params = { "clf__C":[0.01,0.1,1,5,10], "clf__penalty":["l2"] }
log_gs = GridSearchCV(pipe_log, log_params, cv=tscv, scoring="roc_auc", n_jobs=-1)
log_gs.fit(X_train, y_train)
best_log = log_gs.best_estimator_


knn_params = { "clf__n_neighbors":[3,5,7,9,11], "clf__weights":["uniform","distance"] }
knn_gs = GridSearchCV(pipe_knn, knn_params, cv=tscv, scoring="roc_auc", n_jobs=-1)
knn_gs.fit(X_train, y_train)
best_knn = knn_gs.best_estimator_


xgb_params = {
    "clf__n_estimators":[100,200,300],
    "clf__max_depth":[3,5,7],
    "clf__learning_rate":[0.01,0.05,0.1],
    "clf__subsample":[0.8,1.0],
}
xgb_gs = GridSearchCV(pipe_xgb, xgb_params, cv=tscv, scoring="roc_auc", n_jobs=-1)
xgb_gs.fit(X_train, y_train)
best_xgb = xgb_gs.best_estimator_

pipe_log = best_log
pipe_knn = best_knn
pipe_xgb = best_xgb


In [ ]:

# ------------------------------------------------------------
# 7) CALIBRATION (SIGMOID CALIBRATION)
# ------------------------------------------------------------

models = {
    "Logistic": pipe_log,
    "KNN": pipe_knn,
    "XGBoost": pipe_xgb
}


val_split = int(len(X_train) * 0.8)
X_tr, X_val = X_train.iloc[:val_split], X_train.iloc[val_split:]
y_tr, y_val = y_train.iloc[:val_split], y_train.iloc[val_split:]

calibrated_models = {}

print("\n📌 Starting Calibration...\n")

for name, model in models.items():
    print(f"Calibrating {name}...")
    
    model.fit(X_tr, y_tr)
    calib = CalibratedClassifierCV(model, cv="prefit", method="sigmoid")
    calib.fit(X_val, y_val)
    calibrated_models[name] = calib

print("\n✅ Calibration Completed for All Models.\n")

In [ ]:


best_thresholds = {}
thresholds = np.linspace(0.01, 0.99, 99)

print("\n🎯 Finding Best Thresholds...\n")

for name, calib_model in calibrated_models.items():
    val_probs = calib_model.predict_proba(X_val)[:, 1]
    
    f1_scores = [f1_score(y_val, (val_probs >= t).astype(int)) for t in thresholds]
    best_t = thresholds[np.argmax(f1_scores)]
    
    best_thresholds[name] = best_t
    print(f"{name} → Best Threshold: {best_t:.3f}")


In [ ]:

# ------------------------------------------------------------
# 9) FINAL TEST EVALUATION (CALIBRATED + THRESHOLD OPTIMIZED)
# ------------------------------------------------------------


final_results = {}

print("\n📊 Final Model Evaluation...\n")

for name, calib_model in calibrated_models.items():
    t = best_thresholds[name]
    
    test_probs = calib_model.predict_proba(X_test)[:, 1]
    test_pred = (test_probs >= t).astype(int)

    final_results[name] = {
        "Accuracy": accuracy_score(y_test, test_pred),
        "Precision": precision_score(y_test, test_pred, zero_division=0),
        "Recall": recall_score(y_test, test_pred, zero_division=0),
        "F1": f1_score(y_test, test_pred),
        "ROC_AUC": roc_auc_score(y_test, test_probs),
        "Brier Score": brier_score_loss(y_test, test_probs),
        "Threshold": t
    }

final_results_df = pd.DataFrame(final_results).T
print(final_results_df.round(4))

In [ ]:



best_model_name = final_results_df["ROC_AUC"].idxmin()
best_model = calibrated_models[best_model_name]
best_threshold = best_thresholds[best_model_name]

print(f"\n🏆 BEST MODEL SELECTED: {best_model_name}")
print(f"🔑 Best Threshold: {best_threshold:.3f}")

In [ ]:

import joblib

save_dict = {
    "model": best_model,
    "threshold": best_threshold,
    "features": X_train.columns.tolist()
}

joblib.dump(save_dict, "final_rain_model.joblib")

print("\n💾 Final Model Saved as: final_rain_model.joblib")


In [ ]:



# def predict_manual(max_temp, min_temp, mean_temp, humidity, pressure, windspeed,
#                    cloudcover, rainfall, date, rain_lag1=0, rain_3sum=0):
    
#     bundle = joblib.load("final_rain_model.joblib")
#     model = bundle["model"]
#     threshold = bundle["threshold"]
    
#     dt = pd.to_datetime(date)
#     temp_range = max_temp - min_temp
#     month = dt.month
#     doy = dt.dayofyear
#     is_monsoon = 1 if month in [6,7,8,9] else 0

#     X = pd.DataFrame([[
#         max_temp, min_temp, mean_temp, humidity, pressure, windspeed,
#         cloudcover, rainfall, rain_lag1, rain_3sum, temp_range,
#         month, doy, is_monsoon
#     ]], columns=bundle["features"])

#     prob = model.predict_proba(X)[0][1] * 100
#     prediction = "Rain" if prob >= (threshold * 100) else "No Rain"

#     return prediction, prob


